In [11]:
# ============================================================
# TASK 1: WEB SCRAPING
# We're going to scrape book data from books.toscrape.com —
# a website specifically built for scraping practice. Perfect!
# We'll grab: Title, Price, Rating, Availability, and Category.
# ============================================================

# A little helper to convert word-based ratings to numbers
# because the site stores ratings as words like "Three", "Five", etc.
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

def scrape_books_page(url, category_name):
    """
    Scrapes all books from a single page of a given category.
    Returns a list of dictionaries — one per book.
    """
    books_on_this_page = []

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # This will yell at us if something goes wrong (like a 404)
    except requests.exceptions.RequestException as e:
        print(f"   Couldn't reach {url}. Error: {e}")
        return books_on_this_page  # Return empty list and move on gracefully

    soup = BeautifulSoup(response.text, "html.parser")
    book_cards = soup.find_all("article", class_="product_pod")

    for card in book_cards:
        # Pull the title from the 'title' attribute of the <a> tag (it has the full title)
        title = card.h3.a["title"]

        # Price is stored as text like "£12.99" — we'll clean the £ sign later
        price_raw = card.find("p", class_="price_color").text.strip()

        # Rating is a CSS class name like "star-rating Three"
        rating_word = card.find("p", class_="star-rating")["class"][1]
        rating_numeric = rating_map.get(rating_word, 0)

        # Check if the book is actually in stock
        availability = card.find("p", class_="instock availability").text.strip()

        books_on_this_page.append({
            "title": title,
            "price_raw": price_raw,
            "rating": rating_numeric,
            "availability": availability,
            "category": category_name
        })

    return books_on_this_page


def scrape_category(base_url, category_slug, category_name, max_pages=3):
    """
    Scrapes multiple pages from one book category.
    We're limiting to 3 pages per category to keep things reasonable.
    """
    all_books = []
    print(f"  📂 Scraping category: '{category_name}'...")

    for page_num in range(1, max_pages + 1):
        if page_num == 1:
            page_url = f"{base_url}/catalogue/category/books/{category_slug}/index.html"
        else:
            page_url = f"{base_url}/catalogue/category/books/{category_slug}/page-{page_num}.html"

        page_books = scrape_books_page(page_url, category_name)

        if not page_books:
            # No books found means we've probably run out of pages
            break

        all_books.extend(page_books)
        time.sleep(0.5)  # Being polite — let's not hammer the server with requests

    print(f"    → Found {len(all_books)} books in '{category_name}'")
    return all_books


# --- Let's kick off the scraping! ---
BASE_URL = "https://books.toscrape.com"

# We'll scrape a handful of categories to get a diverse dataset
categories_to_scrape = {
    "mystery_3": "Mystery",
    "science-fiction_16": "Science Fiction",
    "romance_8": "Romance",
    "history_32": "History",
    "travel_2": "Travel"
}

print("🚀 Starting the web scraping process...\n")
all_scraped_books = []

for slug, name in categories_to_scrape.items():
    category_books = scrape_category(BASE_URL, slug, name, max_pages=3)
    all_scraped_books.extend(category_books)

print(f"\n Scraping complete! Total books collected: {len(all_scraped_books)}")

🚀 Starting the web scraping process...

  📂 Scraping category: 'Mystery'...
   Couldn't reach https://books.toscrape.com/catalogue/category/books/mystery_3/page-3.html. Error: 404 Client Error: Not Found for url: https://books.toscrape.com/catalogue/category/books/mystery_3/page-3.html
    → Found 32 books in 'Mystery'
  📂 Scraping category: 'Science Fiction'...
   Couldn't reach https://books.toscrape.com/catalogue/category/books/science-fiction_16/page-2.html. Error: 404 Client Error: Not Found for url: https://books.toscrape.com/catalogue/category/books/science-fiction_16/page-2.html
    → Found 16 books in 'Science Fiction'
  📂 Scraping category: 'Romance'...
   Couldn't reach https://books.toscrape.com/catalogue/category/books/romance_8/page-3.html. Error: 404 Client Error: Not Found for url: https://books.toscrape.com/catalogue/category/books/romance_8/page-3.html
    → Found 35 books in 'Romance'
  📂 Scraping category: 'History'...
   Couldn't reach https://books.toscrape.com/ca

In [12]:
# Let's throw all that scraped data into a Pandas DataFrame
# so we can actually work with it properly.

scraped_data_df = pd.DataFrame(all_scraped_books)

print(" Here's a quick peek at what we got:")
print(scraped_data_df.head(10))
print(f"\nShape of our raw data: {scraped_data_df.shape}")

# Save it to a CSV file so we have a local backup
csv_filename = "books_raw_data.csv"
scraped_data_df.to_csv(csv_filename, index=False)
print(f"\n Raw data saved to '{csv_filename}' successfully!")

 Here's a quick peek at what we got:
                                             title price_raw  rating  \
0                                    Sharp Objects   Â£47.82       4   
1                             In a Dark, Dark Wood   Â£19.63       1   
2                              The Past Never Ends   Â£56.50       4   
3                                 A Murder in Time   Â£16.64       1   
4  The Murder of Roger Ackroyd (Hercule Poirot #4)   Â£44.10       4   
5                   The Last Mile (Amos Decker #2)   Â£54.21       2   
6           That Darkness (Gardiner and Renner #1)   Â£13.92       1   
7             Tastes Like Fear (DI Marnie Rome #3)   Â£10.69       1   
8           A Time of Torment (Charlie Parker #14)   Â£48.35       5   
9          A Study in Scarlet (Sherlock Holmes #1)   Â£16.73       2   

  availability category  
0     In stock  Mystery  
1     In stock  Mystery  
2     In stock  Mystery  
3     In stock  Mystery  
4     In stock  Mystery  
5     In stock